In [20]:
import os
import numpy as np
import pickle

ROOT = ""  # <-- SET THIS CORRECTLY
subjects = [f"subj0{i}" for i in range(1, 9)]
MAX_IMAGES = 7000


# -----------------------------
# PROCESS EACH SUBJECT
# -----------------------------
for subj in subjects:
    print("\n==============================")
    print("Processing:", subj)
    
    subj_path = os.path.join(ROOT, subj)
    
    # -----------------------------
    # LOAD FMRI
    # -----------------------------
    try:
        lh = np.load(os.path.join(subj_path, "training_split", "training_fmri", "lh_training_fmri.npy"))
        rh = np.load(os.path.join(subj_path, "training_split", "training_fmri", "rh_training_fmri.npy"))
    except:
        print("❌ Skipping (missing fMRI):", subj)
        continue
    
    # -----------------------------
    # LOAD IMAGES
    # -----------------------------
    images_dir = os.path.join(subj_path, "training_split", "training_images")
    
    images = sorted([
        f for f in os.listdir(images_dir)
        if f.endswith(".png") and not f.startswith(".")
    ])
    
    num = min(MAX_IMAGES, len(images))
    
    lh = lh[:num]
    rh = rh[:num]
    images = images[:num]
    
    image_paths = [os.path.join(images_dir, img) for img in images]
    
    # -----------------------------
    # ROI PROCESSING
    # -----------------------------
    roi_dir = os.path.join(subj_path, "roi_masks")
    
    subject_dataset = {}
    
    for file in os.listdir(roi_dir):
        if file.startswith("."):
            continue
        
        if not file.endswith("_challenge_space.npy"):
            continue
        
        if "mapping" in file or "all-vertices" in file:
            continue
        
        hemi = file[:2]  # 'lh' or 'rh'
        
        # 🔥 CORRECT FIX (THIS WAS THE BUG)
        # lh.streams_challenge_space.npy → streams
        class_part = file.split(".")[1]
        class_name = class_part.split("_challenge_space")[0]
        
        mapping_path = os.path.join(roi_dir, f"mapping_{class_name}.npy")
        
        if not os.path.exists(mapping_path):
            print("⚠️ Mapping missing for:", file)
            continue
        
        # load mask + mapping
        mask = np.load(os.path.join(roi_dir, file))
        mapping = np.load(mapping_path, allow_pickle=True).item()
        
        # -----------------------------
        # EXTRACT ROIs
        # -----------------------------
        for label, roi_name in mapping.items():
            if label == 0:
                continue
            
            indices = np.where(mask == label)[0]
            
            if len(indices) == 0:
                continue
            
            if hemi == "lh":
                fmri_roi = lh[:, indices]
            else:
                fmri_roi = rh[:, indices]
            
            subject_dataset[roi_name] = {
                "fmri": fmri_roi,
                "image_paths": image_paths
            }
            
            print(f"{roi_name}: {fmri_roi.shape}")
    
    # -----------------------------
    # SAVE
    # -----------------------------
    if len(subject_dataset) == 0:
        print("❌ No ROI extracted for", subj)
        continue
    
    save_path = f"{subj}_roi_dataset.pkl"
    
    with open(save_path, "wb") as f:
        pickle.dump(subject_dataset, f)
    
    print("✅ Saved:", save_path)


Processing: subj01
early: (7000, 5169)
midventral: (7000, 867)
midlateral: (7000, 1091)
midparietal: (7000, 853)
ventral: (7000, 4974)
lateral: (7000, 3129)
parietal: (7000, 2346)
OWFA: (7000, 317)
VWFA-1: (7000, 1381)
VWFA-2: (7000, 461)
mfs-words: (7000, 490)
OPA: (7000, 2806)
PPA: (7000, 891)
RSC: (7000, 660)
OWFA: (7000, 590)
VWFA-1: (7000, 397)
VWFA-2: (7000, 431)
OFA: (7000, 432)
FFA-1: (7000, 552)
EBA: (7000, 2837)
FBA-1: (7000, 574)
OFA: (7000, 305)
FFA-1: (7000, 330)
FFA-2: (7000, 629)
OPA: (7000, 1863)
PPA: (7000, 1311)
RSC: (7000, 401)
EBA: (7000, 3400)
FBA-1: (7000, 206)
FBA-2: (7000, 856)
early: (7000, 5304)
midventral: (7000, 1050)
midlateral: (7000, 1191)
midparietal: (7000, 1133)
ventral: (7000, 4852)
lateral: (7000, 3578)
parietal: (7000, 2314)
V1v: (7000, 710)
V1d: (7000, 828)
V2v: (7000, 632)
V2d: (7000, 692)
V3v: (7000, 567)
V3d: (7000, 669)
hV4: (7000, 531)
V1v: (7000, 444)
V1d: (7000, 991)
V2v: (7000, 887)
V2d: (7000, 725)
V3v: (7000, 682)
V3d: (7000, 535)
hV4: (

In [21]:
import os
import numpy as np
import pandas as pd
import pickle
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


DATA_DIR = "."   
N_COMPONENTS = 200

files = [
    f for f in os.listdir(DATA_DIR)
    if f.endswith("_roi_dataset.pkl")
    and not f.startswith(".")
    and not f.startswith("._")
]

for file in files:

    print("Processing:", file)
    
    with open(os.path.join(DATA_DIR, file), "rb") as f:
        subject_dataset = pickle.load(f)
    
    subject_name = file.split("_")[0]
    
    pca_tables = {}
    pca_info = {}
    
    for roi, data in subject_dataset.items():
        print("\nROI:", roi)
        
        fmri = data["fmri"]
        image_paths = data["image_paths"]
        
        print("Original shape:", fmri.shape)
        
        scaler = StandardScaler()
        fmri_norm = scaler.fit_transform(fmri)

        n_comp = min(N_COMPONENTS, fmri_norm.shape[1])
        
        pca = PCA(n_components=n_comp)
        fmri_pca = pca.fit_transform(fmri_norm)
        
        print("PCA shape:", fmri_pca.shape)
        
        df = pd.DataFrame(fmri_pca, columns=[f"PC{i+1}" for i in range(n_comp)])
        df["image_path"] = image_paths
        df["subject"] = subject_name
        df["roi"] = roi
        
        cols = ["subject", "roi", "image_path"] + [f"PC{i+1}" for i in range(n_comp)]
        df = df[cols]
        
        pca_tables[roi] = df

        pca_info[roi] = {
            "components": pca.components_,
            "explained_variance": pca.explained_variance_ratio_
        }
    
    out_tables = f"{subject_name}_pca_tables.pkl"
    out_info = f"{subject_name}_pca_info.pkl"
    
    with open(out_tables, "wb") as f:
        pickle.dump(pca_tables, f)
    
    with open(out_info, "wb") as f:
        pickle.dump(pca_info, f)
    
    print(f"\nSaved: {out_tables}, {out_info}")

Processing: subj01_fsaverage_roi_dataset.pkl

Saved: subj01_pca_tables.pkl, subj01_pca_info.pkl
Processing: subj02_fsaverage_roi_dataset.pkl

Saved: subj02_pca_tables.pkl, subj02_pca_info.pkl
Processing: subj03_fsaverage_roi_dataset.pkl

Saved: subj03_pca_tables.pkl, subj03_pca_info.pkl
Processing: subj04_fsaverage_roi_dataset.pkl

Saved: subj04_pca_tables.pkl, subj04_pca_info.pkl
Processing: subj05_fsaverage_roi_dataset.pkl

Saved: subj05_pca_tables.pkl, subj05_pca_info.pkl
Processing: subj06_fsaverage_roi_dataset.pkl

Saved: subj06_pca_tables.pkl, subj06_pca_info.pkl
Processing: subj07_fsaverage_roi_dataset.pkl

Saved: subj07_pca_tables.pkl, subj07_pca_info.pkl
Processing: subj08_fsaverage_roi_dataset.pkl

Saved: subj08_pca_tables.pkl, subj08_pca_info.pkl
Processing: subj01_roi_dataset.pkl

ROI: early
Original shape: (7000, 5304)
PCA shape: (7000, 200)

ROI: midventral
Original shape: (7000, 1050)
PCA shape: (7000, 200)

ROI: midlateral
Original shape: (7000, 1191)
PCA shape: (7000, 

In [22]:
import os
import pickle

DATA_DIR = "."  
TOP_K = 20

files = [
    f for f in os.listdir(DATA_DIR)
    if f.endswith("_pca_tables.pkl")
    and not f.startswith(".")
    and not f.startswith("._")
]


for file in files:
    print("Processing:", file)
    
    with open(os.path.join(DATA_DIR, file), "rb") as f:
        pca_tables = pickle.load(f)
    
    subject_name = file.split("_")[0]
    
    top_images = {}
    
    for roi, df in pca_tables.items():
        print("\nROI:", roi)
        
        roi_result = {}
        
        pc_cols = [col for col in df.columns if col.startswith("PC")]
        
        for pc in pc_cols:
            sorted_df = df.sort_values(by=pc, ascending=False)
            
            top_k_df = sorted_df.head(TOP_K)
            
            roi_result[pc] = {
                "image_paths": top_k_df["image_path"].tolist(),
                "scores": top_k_df[pc].tolist()
            }
        
        top_images[roi] = roi_result
    
    out_file = f"{subject_name}_top_images.pkl"
    
    with open(out_file, "wb") as f:
        pickle.dump(top_images, f)
    
    print(f"\nSaved: {out_file}")

Processing: subj01_pca_tables.pkl

ROI: early

ROI: midventral

ROI: midlateral

ROI: midparietal

ROI: ventral

ROI: lateral

ROI: parietal

ROI: OWFA

ROI: VWFA-1

ROI: VWFA-2

ROI: mfs-words

ROI: OPA

ROI: PPA

ROI: RSC

ROI: OFA

ROI: FFA-1

ROI: EBA

ROI: FBA-1

ROI: FFA-2

ROI: FBA-2

ROI: V1v

ROI: V1d

ROI: V2v

ROI: V2d

ROI: V3v

ROI: V3d

ROI: hV4

Saved: subj01_top_images.pkl
Processing: subj02_pca_tables.pkl

ROI: early

ROI: midventral

ROI: midlateral

ROI: midparietal

ROI: ventral

ROI: lateral

ROI: parietal

ROI: OWFA

ROI: VWFA-1

ROI: VWFA-2

ROI: mfs-words

ROI: OPA

ROI: PPA

ROI: RSC

ROI: OFA

ROI: FFA-1

ROI: FFA-2

ROI: EBA

ROI: FBA-2

ROI: V1v

ROI: V1d

ROI: V2v

ROI: V2d

ROI: V3v

ROI: V3d

ROI: hV4

Saved: subj02_top_images.pkl
Processing: subj03_pca_tables.pkl

ROI: early

ROI: midventral

ROI: midlateral

ROI: midparietal

ROI: ventral

ROI: lateral

ROI: parietal

ROI: OWFA

ROI: VWFA-1

ROI: VWFA-2

ROI: mfs-words

ROI: OPA

ROI: PPA

ROI: RSC

ROI

In [3]:
!pip3 install transformers

     |████████████████████████████████| 10.2 MB 2.0 MB/s eta 0:00:01
     |████████████████████████████████| 3.0 MB 15.2 MB/s eta 0:00:01
     |████████████████████████████████| 637 kB 23.5 MB/s eta 0:00:01
     |████████████████████████████████| 447 kB 30.8 MB/s eta 0:00:01
     |████████████████████████████████| 56 kB 5.2 MB/s  eta 0:00:01
     |████████████████████████████████| 3.6 MB 18.6 MB/s eta 0:00:01
     |████████████████████████████████| 310 kB 19.0 MB/s eta 0:00:01
     |████████████████████████████████| 87 kB 21.1 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Frameworks/Python.framework/Versions/3.10/bin/python3.10 -m pip install --upgrade pip' command.


In [2]:
import os
import pickle
import torch
import numpy as np
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

DATA_DIR = "."
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"

print("Using device:", DEVICE)

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

model.eval()


files = [
    f for f in os.listdir(DATA_DIR)
    if f.endswith("_top_images.pkl")
    and not f.startswith(".")
    and not f.startswith("._")
]

print("Found files:", files)


@torch.no_grad()
def get_embeddings(image_paths):
    images = []
    
    for p in image_paths:
        try:
            img = Image.open(p).convert("RGB")
            images.append(img)
        except:
            continue
    
    if len(images) < 2:
        return None
    
    inputs = processor(images=images, return_tensors="pt", padding=True).to(DEVICE)
    
    outputs = model.get_image_features(**inputs)
    
    # ✅ FIX
    if hasattr(outputs, "pooler_output"):
        features = outputs.pooler_output
    else:
        features = outputs
    
    # normalize
    features = features / features.norm(dim=-1, keepdim=True)
    
    return features.cpu().numpy()

for file in files:
    print("Processing:", file)
    
    with open(os.path.join(DATA_DIR, file), "rb") as f:
        data = pickle.load(f)
    
    subject_name = file.split("_")[0]
    
    results = {}
    
    for roi, roi_data in data.items():
        print("\nROI:", roi)
        
        roi_scores = {}

        for pc, pc_data in roi_data.items():
            image_paths = pc_data["image_paths"]
            
            if len(image_paths) < 2:
                continue
            
            emb = get_embeddings(image_paths)  
            
            sim_matrix = emb @ emb.T  
            
            n = sim_matrix.shape[0]
            i_upper = np.triu_indices(n, k=1)
            
            pairwise_scores = sim_matrix[i_upper]
            
            avg_score = float(pairwise_scores.mean())
            
            roi_scores[pc] = avg_score
        
        results[roi] = roi_scores
    
    out_file = f"{subject_name}_clip_similarity.pkl"
    
    with open(out_file, "wb") as f:
        pickle.dump(results, f)
    
    print(f"\nSaved: {out_file}")

Using device: mps


Loading weights: 100%|███████████████████████| 398/398 [00:00<00:00, 54966.51it/s]
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Found files: ['subj01_top_images.pkl', 'subj02_top_images.pkl', 'subj03_top_images.pkl', 'subj04_top_images.pkl', 'subj05_top_images.pkl', 'subj06_top_images.pkl', 'subj07_top_images.pkl', 'subj08_top_images.pkl']
Processing: subj01_top_images.pkl

ROI: early

ROI: midventral

ROI: midlateral

ROI: midparietal

ROI: ventral

ROI: lateral

ROI: parietal

ROI: OWFA

ROI: VWFA-1

ROI: VWFA-2

ROI: mfs-words

ROI: OPA

ROI: PPA

ROI: RSC

ROI: OFA

ROI: FFA-1

ROI: EBA

ROI: FBA-1

ROI: FFA-2

ROI: FBA-2

ROI: V1v

ROI: V1d

ROI: V2v

ROI: V2d

ROI: V3v

ROI: V3d

ROI: hV4

Saved: subj01_clip_similarity.pkl
Processing: subj02_top_images.pkl

ROI: early

ROI: midventral

ROI: midlateral

ROI: midparietal

ROI: ventral

ROI: lateral

ROI: parietal

ROI: OWFA

ROI: VWFA-1

ROI: VWFA-2

ROI: mfs-words

ROI: OPA

ROI: PPA

ROI: RSC

ROI: OFA

ROI: FFA-1

ROI: FFA-2

ROI: EBA

ROI: FBA-2

ROI: V1v

ROI: V1d

ROI: V2v

ROI: V2d

ROI: V3v

ROI: V3d

ROI: hV4

Saved: subj02_clip_similarity.pkl
Proc

In [4]:
import os
import pickle

DATA_DIR = "."

files = [
    f for f in os.listdir(DATA_DIR)
    if f.endswith("_clip_similarity.pkl")
    and not f.startswith(".")
    and not f.startswith("._")
]

print("Found files:", files)


for file in files:

    print("Processing:", file)
    
    with open(os.path.join(DATA_DIR, file), "rb") as f:
        data = pickle.load(f)
    
    subject_name = file.split("_")[0]
    
    top40_results = {}

    for roi, pc_scores in data.items():
        
        if len(pc_scores) == 0:
            continue
        
        sorted_pcs = sorted(pc_scores.items(), key=lambda x: x[1], reverse=True)
        
        top_pcs = sorted_pcs[:40]
        

        top40_results[roi] = dict(top_pcs)
        
        print(f"{roi}: kept {len(top_pcs)} PCs")
    
    out_file = f"{subject_name}_top40_clip.pkl"
    
    with open(out_file, "wb") as f:
        pickle.dump(top40_results, f)
    
    print("✅ Saved:", out_file)

Found files: ['subj01_clip_similarity.pkl', 'subj02_clip_similarity.pkl', 'subj03_clip_similarity.pkl', 'subj04_clip_similarity.pkl', 'subj05_clip_similarity.pkl', 'subj06_clip_similarity.pkl', 'subj07_clip_similarity.pkl', 'subj08_clip_similarity.pkl']
Processing: subj01_clip_similarity.pkl
early: kept 40 PCs
midventral: kept 40 PCs
midlateral: kept 40 PCs
midparietal: kept 40 PCs
ventral: kept 40 PCs
lateral: kept 40 PCs
parietal: kept 40 PCs
OWFA: kept 40 PCs
VWFA-1: kept 40 PCs
VWFA-2: kept 40 PCs
mfs-words: kept 40 PCs
OPA: kept 40 PCs
PPA: kept 40 PCs
RSC: kept 40 PCs
OFA: kept 40 PCs
FFA-1: kept 40 PCs
EBA: kept 40 PCs
FBA-1: kept 40 PCs
FFA-2: kept 40 PCs
FBA-2: kept 40 PCs
V1v: kept 40 PCs
V1d: kept 40 PCs
V2v: kept 40 PCs
V2d: kept 40 PCs
V3v: kept 40 PCs
V3d: kept 40 PCs
hV4: kept 40 PCs
✅ Saved: subj01_top40_clip.pkl
Processing: subj02_clip_similarity.pkl
early: kept 40 PCs
midventral: kept 40 PCs
midlateral: kept 40 PCs
midparietal: kept 40 PCs
ventral: kept 40 PCs
lateral

In [3]:
!pip3 install ollama


You should consider upgrading via the '/Library/Frameworks/Python.framework/Versions/3.10/bin/python3.10 -m pip install --upgrade pip' command.


In [ ]:
import os
import pickle
import json
import ollama
from concurrent.futures import ThreadPoolExecutor

# ==============================
# CONFIG
# ==============================

MODEL        = "llava"
MAX_WORKERS  = 2   # 2 parallel jobs is safe for 8GB VRAM with llava (~4-5GB)
BATCH_SIZE   = 5   # images per batch within each PC call (20 images -> 4 batches)

DATA_ROOT  = r"E:\ML PROJECT COURSE"
BASE_DIR   = DATA_ROOT
CACHE_FILE = "image_cache.json"

SUBJECTS = [f"subj0{i}" for i in range(1, 9)]  # subj01 through subj08

PROMPT = """You are given multiple images representing a brain's response to a visual concept.

Generate EXACTLY 5 single words that best capture the shared visual theme across ALL images.

Respond ONLY in this format, nothing else:
1. word
2. word
3. word
4. word
5. word

Rules:
- Every word must be singular (e.g. "motorcycle" not "motorcycles", "face" not "faces")
- Use broad, general concepts not specific instances (e.g. "food" not "pasta", "vehicle" not "Ferrari", "animal" not "labrador")
- Each word must be a single noun or adjective — no phrases, no hyphens
- Words must represent themes visible across ALL images, not just one
- No repetition — all 5 words must be distinct
"""

# ==============================
# FIND PKL FILES
# ==============================

PKL_FILES = [
    os.path.join(BASE_DIR, f"{subj}_top_images.pkl")
    for subj in SUBJECTS
    if os.path.exists(os.path.join(BASE_DIR, f"{subj}_top_images.pkl"))
]
print(f"Found {len(PKL_FILES)} PKL files:")
for p in PKL_FILES:
    print(f"  {os.path.basename(p)}")

# ==============================
# UTILS
# ==============================

def save_json_atomic(data, path):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(data, f, indent=2)
    os.replace(tmp, path)

def load_cache(path):
    if os.path.exists(path):
        with open(path, "r") as f:
            cache = json.load(f)
        print(f"[OK] Loaded cache ({len(cache)} entries)")
        return cache
    return {}

def get_entries_for_subject(data, top40_clip):
    """
    PKL structure: data[roi_group][pc_name] = {'image_paths': [...], 'scores': [...]}
    top40_clip structure: top40_clip[roi_group][pc_name] = score (float)
    Returns flat dict: {'early__PC1': [abs_path, ...], ...}
    Only includes the top-40 PCs per ROI as specified in top40_clip.
    """
    entries = {}
    for roi_group, pc_dict in data.items():
        if not isinstance(pc_dict, dict):
            continue
        # Only keep the PCs that appear in top40_clip for this ROI
        allowed_pcs = set(top40_clip.get(roi_group, {}).keys())
        for pc_name, pc_value in pc_dict.items():
            if pc_name not in allowed_pcs:
                continue
            if not isinstance(pc_value, dict):
                continue
            rel_paths = pc_value.get("image_paths", [])
            valid = []
            for rel in rel_paths:
                # Normalize slashes so Mac-style paths work on Windows too
                rel = rel.replace("/", os.sep)
                if rel.startswith("."):
                    continue
                abs_path = os.path.join(DATA_ROOT, rel)
                if os.path.exists(abs_path):
                    valid.append(abs_path)
            if valid:
                entries[f"{roi_group}__{pc_name}"] = valid
    return entries

def parse_captions(text):
    """Parse 5 single words from numbered list or comma-separated response. Returns None if fewer than 5 found."""
    text = text.strip()
    words = []

    # Try numbered list first (e.g. "1. word" or "1. two words")
    for line in text.split("\n"):
        line = line.strip()
        if not line:
            continue
        if line[0].isdigit():
            # Strip leading number and punctuation
            phrase = line.lstrip("0123456789").lstrip(".)- ").strip()
            if not phrase:
                continue
            # If multi-word phrase, just take the first meaningful word
            word = phrase.split()[0].strip(".,;:-")
            if word:
                words.append(word)

    # Fall back to comma-separated (e.g. "pink, water, boat" or "Soccer goalkeeper, Tennis balls")
    if len(words) < 5:
        candidates = [w.strip().strip(".,;") for w in text.split(",")]
        words = []
        for phrase in candidates:
            phrase = phrase.strip()
            if not phrase:
                continue
            # Take first word of each comma chunk
            word = phrase.split()[0].strip(".,;:-")
            if word:
                words.append(word)

    # Last resort: just grab any words from the whole text
    if len(words) < 5:
        all_tokens = [t.strip(".,;:-()[]") for t in text.split()]
        words = [t for t in all_tokens if t and t[0].isalpha()]

    if len(words) < 5:
        return None, text  # truly unparseable
    return words[:5], None

def caption_pc(images, retries=3):
    """
    Split images into batches, get 5 words per batch, then pick the 5
    most frequent words across all batches as the final captions.
    """
    batches = [images[i:i + BATCH_SIZE] for i in range(0, len(images), BATCH_SIZE)]
    all_words = []

    for batch_idx, batch in enumerate(batches):
        success = False
        for attempt in range(1, retries + 1):
            try:
                response = ollama.chat(
                    model=MODEL,
                    messages=[{"role": "user", "content": PROMPT, "images": batch}]
                )
                captions, raw = parse_captions(response["message"]["content"])
                if captions is not None:
                    all_words.extend(captions)
                    success = True
                    break
                print(f"  [WARN] Batch {batch_idx+1} bad parse attempt {attempt}, raw: {raw!r}")
            except Exception as e:
                print(f"  [WARN] Batch {batch_idx+1} error attempt {attempt}: {e}")
        if not success:
            print(f"  [FAIL] Batch {batch_idx+1} failed after {retries} attempts, skipping batch")

    if not all_words:
        return None

    # Pick the 5 most frequent words across all batches
    from collections import Counter
    counts = Counter(w.lower() for w in all_words)
    top5 = [word for word, _ in counts.most_common(5)]

    # Pad with remaining unique words if somehow fewer than 5
    if len(top5) < 5:
        extras = [w for w in all_words if w.lower() not in top5]
        top5 += extras[:5 - len(top5)]

    return top5[:5]

# ==============================
# PROCESS ONE SUBJECT
# ==============================

def process_subject(pkl_file, global_cache):
    try:
        with open(pkl_file, "rb") as f:
            data = pickle.load(f)
    except Exception as e:
        print(f"⚠ Skipping corrupted file: {pkl_file} -- {e}")
        return

    subject_name = os.path.basename(pkl_file).split("_")[0]

    # Load the corresponding top40_clip pkl
    top40_clip_file = os.path.join(BASE_DIR, f"{subject_name}_top40_clip.pkl")
    try:
        with open(top40_clip_file, "rb") as f:
            top40_clip = pickle.load(f)
    except Exception as e:
        print(f"[WARN] Could not load top40_clip for {subject_name}: {e}")
        return

    print(f"\n[START] Processing {subject_name}")

    subject_output_file = f"captions_{subject_name}.json"
    if os.path.exists(subject_output_file):
        with open(subject_output_file, "r") as f:
            subject_results = json.load(f)
        print(f"  [RESUME] ({len(subject_results)} PCs already done)")
    else:
        subject_results = {}

    entries = get_entries_for_subject(data, top40_clip)
    print(f"  [INFO] {len(entries)} PCs to process")

    def process_one(item):
        key, images = item
        if key in subject_results:
            return key, None  # already done, skip

        cache_key = "|".join(sorted(images))
        if cache_key in global_cache:
            print(f"  [CACHE] {key}")
            return key, global_cache[cache_key]

        print(f"  [RUN] {key} | {len(images)} images -> 5 captions")
        captions = caption_pc(images)
        if captions:
            global_cache[cache_key] = captions
        return key, captions

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        for key, captions in executor.map(process_one, entries.items()):
            if captions is None:
                continue
            subject_results[key] = {
                "images": entries[key],
                "captions": captions
            }
            save_json_atomic(subject_results, subject_output_file)
            save_json_atomic(global_cache, CACHE_FILE)

    print(f"  [DONE] {subject_name}")

# ==============================
# RUN
# ==============================

global_cache = load_cache(CACHE_FILE)

for pkl_file in PKL_FILES:
    process_subject(pkl_file, global_cache)

save_json_atomic(global_cache, CACHE_FILE)
print("\n[ALL DONE]")
print(f"[INFO] Total cached PCs: {len(global_cache)}")

Found 8 PKL files:
  subj01_top_images.pkl
  subj02_top_images.pkl
  subj03_top_images.pkl
  subj04_top_images.pkl
  subj05_top_images.pkl
  subj06_top_images.pkl
  subj07_top_images.pkl
  subj08_top_images.pkl

[START] Processing subj01
  [INFO] 1080 PCs to process
  [RUN] early__PC3 | 20 images -> 5 captions
  [RUN] early__PC4 | 20 images -> 5 captions
  [WARN] Batch 2 bad parse attempt 1, raw: '[words: 1. Tennis racket'
  [WARN] Batch 2 bad parse attempt 1, raw: '1. Bird'
  [RUN] early__PC8 | 20 images -> 5 captions
  [WARN] Batch 4 bad parse attempt 1, raw: 'Time'
  [WARN] Batch 1 bad parse attempt 1, raw: 'Home remodel\n\n1. Man'
  [RUN] early__PC15 | 20 images -> 5 captions
  [WARN] Batch 3 bad parse attempt 1, raw: "[children's bike"
  [RUN] early__PC17 | 20 images -> 5 captions
  [WARN] Batch 4 bad parse attempt 1, raw: '1. Elephant'


In [1]:
import json

FILE = "captions_subj01.json"

# =========================
# LOAD
# =========================
with open(FILE, "r") as f:
    data = json.load(f)

print("Total entries:", len(data))

# =========================
# CHECKS
# =========================
missing_captions = []
wrong_length = []
bad_values = []

for key, val in data.items():

    # check captions key
    if "captions" not in val:
        missing_captions.append(key)
        continue

    caps = val["captions"]

    # check type
    if not isinstance(caps, list):
        bad_values.append((key, caps))
        continue

    # check length
    if len(caps) != 5:
        wrong_length.append((key, caps))
        continue

    # check content
    for w in caps:
        if not isinstance(w, str) or not w.strip():
            bad_values.append((key, caps))
            break

# =========================
# REPORT
# =========================
print("\n===== REPORT =====")

print("❌ Missing captions:", len(missing_captions))
print("❌ Wrong length (!=5):", len(wrong_length))
print("❌ Bad values:", len(bad_values))

# samples
if wrong_length:
    print("\nSample wrong length:")
    for item in wrong_length[:5]:
        print(item)

if bad_values:
    print("\nSample bad values:")
    for item in bad_values[:5]:
        print(item)

# =========================
# ROI-wise breakdown
# =========================
print("\n===== ROI BREAKDOWN =====")

roi_counts = {}

for key in data:
    roi = key.split("__")[0]
    roi_counts.setdefault(roi, 0)
    roi_counts[roi] += 1

for roi, count in roi_counts.items():
    print(f"{roi}: {count} PCs")

Total entries: 1080

===== REPORT =====
❌ Missing captions: 0
❌ Wrong length (!=5): 0
❌ Bad values: 0

===== ROI BREAKDOWN =====
early: 40 PCs
midventral: 40 PCs
midlateral: 40 PCs
midparietal: 40 PCs
ventral: 40 PCs
lateral: 40 PCs
parietal: 40 PCs
OWFA: 40 PCs
VWFA-1: 40 PCs
VWFA-2: 40 PCs
mfs-words: 40 PCs
OPA: 40 PCs
PPA: 40 PCs
RSC: 40 PCs
OFA: 40 PCs
FFA-1: 40 PCs
EBA: 40 PCs
FBA-1: 40 PCs
FFA-2: 40 PCs
FBA-2: 40 PCs
V1v: 40 PCs
V1d: 40 PCs
V2v: 40 PCs
V2d: 40 PCs
V3v: 40 PCs
V3d: 40 PCs
hV4: 40 PCs
